# EdgeGuardian — Colab GPU functional test

This notebook validates the *real* perception + agent stack on a free GPU (T4/L4) before the GB10 run. It is a portability check only: the final AI workload must run on the HP ZGX Nano.

1. Runtime → Change runtime type → GPU.
2. Upload `edge-guardian.zip` (File browser → upload) or clone your public repo.

In [ ]:
!nvidia-smi
import os, sys
if not os.path.exists('edge-guardian'):
    if os.path.exists('edge-guardian.zip'):
        !unzip -q edge-guardian.zip
    else:
        !git clone https://github.com/<your-user>/edge-guardian.git
%cd edge-guardian
!pip -q install -r requirements/base.txt -r requirements/vision.txt -r requirements/audio.txt
!pip -q install -e . --no-deps

## 1. Software tests (fixture profile, no models)

In [ ]:
!python scripts/make_synthetic_scenarios.py --out tests/.fixtures
!python -m pytest -q

## 2. Download pretrained models

In [ ]:
!python scripts/download_models.py --asr medium

## 3. Serve a local LLM/VLM with Ollama (same OpenAI-compatible API as ZRT/vLLM on the GB10)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time
subprocess.Popen(['ollama','serve'])
time.sleep(5)
!ollama pull qwen3.5:9b
!ollama pull nomic-embed-text
!curl -s http://127.0.0.1:11434/v1/models

## 4. Real-media test: UR Fall Detection clip (CC BY-NC-SA 4.0) through the full stack

In [ ]:
!python scripts/download_demo_media.py --set urfd --n 2
!python -m edge_guardian analyze data/downloads/urfd/fall-01-cam0.mp4 --profile colab \
    --ask "Did anyone fall or get hurt? Decide what to do." --answer "" --run-name colab-urfd-fall

In [ ]:
import json, glob
r = json.load(open(sorted(glob.glob('runs/colab-urfd-fall*/report.json'))[-1]))
print(r['final_answer'])
print('events:', [(e['kind'], round(e['start_s'],1), round(e['end_s'],1), round(e['confidence'],2)) for e in r['events']])
print('executed:', r['executed_actions'])
print('telemetry:', {k: r['telemetry'][k] for k in ('wall_time_s','model_calls','input_tokens','output_tokens','frames_processed','frames_total')})

## 5. Benchmark + ablations on the synthetic suite with the real LLM as judge

In [ ]:
!python scripts/benchmark.py --profile colab --variants full no_critic no_ask_user fixed_pipeline

## 6. Web UI (optional): expose port 8600 with a tunnel of your choice

In [ ]:
# !python -m edge_guardian serve --profile colab --host 0.0.0.0 --port 8600